# Text Database

In [1]:
texts = [
    "AI models are getting larger every year",
    "Coffee prices fluctuate with global demand",
    "Python loops make repetitive tasks easier",
    "SQL joins merge data across related tables",
    "The best time to email customers is afternoon",
    "Vector databases improve semantic search",
    "Transfer learning reduces training time",
    "Open source drives faster innovation",
    "Cosine similarity measures text closeness",
    "Apple launches new devices every September",
    "Pandas groupby aggregates large datasets",
    "Feature engineering improves model accuracy",
    "Normalization helps models converge faster",
    "Random forests combine many decision trees",
    "Gradient boosting refines weak learners",
    "K-means clustering groups similar points",
    "Dimensionality reduction speeds up training",
    "Visualization helps explain complex data",
    "Streaming data needs real-time processing",
    "Outlier detection prevents model bias",
    "Cross-validation improves generalization",
    "Hyperparameter tuning optimizes accuracy",
    "Batch processing handles large datasets",
    "Cloud storage scales automatically",
    "Data pipelines automate ETL workflows",
    "Machine learning powers recommendation systems",
    "Text embeddings convert words into vectors",
    "Image embeddings capture visual similarity",
    "Speech recognition converts sound to text",
    "Transformers replaced RNNs in NLP",
    "Attention mechanism focuses on context",
    "Reinforcement learning learns by reward",
    "Synthetic data helps balance datasets",
    "Time series forecasting predicts demand",
    "Business intelligence helps decision making",
    "Interactive dashboards improve insights",
    "A/B testing measures campaign success",
    "Customer segmentation improves marketing",
    "Fraud detection protects financial systems",
    "Personalization improves customer experience",
    "Deep learning requires large datasets",
    "Model evaluation ensures reliability",
    "Explainability builds model trust",
    "Ethics in AI ensures responsible innovation",
    "Data governance ensures data quality",
    "API integration connects multiple systems",
    "Microservices improve scalability",
    "Docker containers simplify deployment",
    "CI/CD pipelines automate software releases",
    "Serverless computing optimizes cost",
    "Edge AI brings inference closer to devices",
    "Federated learning protects privacy",
    "Prompt engineering improves LLM responses",
    "Tokenization breaks text into subwords",
    "Embeddings capture semantic meaning",
    "Vector search retrieves similar content",
    "RAG combines retrieval with generation",
    "LangChain simplifies LLM orchestration",
    "Agents coordinate multiple AI tasks",
    "Pinecone indexes large vector datasets",
    "Chroma provides lightweight vector storage",
    "Faiss accelerates approximate nearest neighbor search",
    "Sentence transformers create text embeddings",
    "Fine-tuning customizes pretrained models",
    "Quantization reduces model size",
    "Distillation transfers knowledge to smaller models",
    "LoRA fine-tunes large models efficiently",
    "RLHF aligns models with human feedback",
    "GPT models use autoregressive generation",
    "BERT uses bidirectional attention",
    "LLMs understand context and nuance",
    "OpenAI advances general-purpose AI",
    "Google trains foundation models at scale",
    "Meta focuses on open research models",
    "Anthropic emphasizes safety and alignment",
    "NVIDIA optimizes GPUs for deep learning",
    "Transformer architecture scales efficiently",
    "Self-attention connects distant tokens",
    "Batch normalization stabilizes learning",
    "Dropout prevents overfitting",
    "ReLU activation speeds convergence",
    "Adam optimizer adapts learning rates",
    "Learning rate scheduling improves training",
    "Data augmentation improves robustness",
    "Model checkpoints save progress",
    "Early stopping prevents overtraining",
    "Evaluation metrics guide improvement",
    "Precision and recall measure performance",
    "ROC curve visualizes trade-offs",
    "Confusion matrix summarizes predictions",
    "F1 score balances precision and recall",
    "AUC measures classification quality",
    "Regression minimizes squared errors",
    "Classification predicts categorical outcomes",
    "Clustering groups unlabeled data",
    "Anomaly detection identifies unusual patterns",
    "Dimensionality reduction simplifies data visualization",
    "Principal component analysis reduces redundancy",
    "t-SNE projects data into 2D space",
    "UMAP preserves local relationships",
    "Word2Vec learns from co-occurrence statistics",
    "Doc2Vec represents full sentences as vectors",
    "TF-IDF weighs word importance in documents",
    "BM25 scores document relevance in search engines",
]

# Creating imports

In [35]:
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from pinecone import Pinecone, ServerlessSpec

### loading api keys

In [37]:
load_dotenv()
gemini_api_key = os.getenv("GEMINI_API_KEY")
pinecone_key = os.getenv("PINECONE_KEY")

# Create dense embeddings

In [21]:

model = SentenceTransformer('all-MiniLM-L6-v2')
dense_embeddings = model.encode(texts)
print(len(texts))
dense_embeddings.shape


104


(104, 384)

# Create Sparse Vector Embeddings

In [ ]:
vectorizer = TfidfVectorizer()
sparse_embeddings = vectorizer.fit_transform(texts)

[0.3203649566775846,
 0.27799258044110875,
 0.41149353572237,
 0.41149353572237,
 0.41149353572237,
 0.37786068859762806,
 0.41149353572237]

# Prepare Data

In [55]:
data = []
for i, text in enumerate(texts):
    row = sparse_embeddings[i]
    coo = row.tocoo()
    sparse_data = {
        "indices": coo.col.tolist(),
        "values": coo.data.tolist(),
    }
    data.append({
        "id": str(i),
        "values": dense_embeddings[i].tolist(),
        "sparse_values": sparse_data,
        "metadata": {
            "text": text
        }
    })

# Create Pinecone client

In [61]:
pc = Pinecone(api_key=pinecone_key)

index_name = "rag-practice-easy-2"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="dotproduct",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1",
        ),
    )

pc_index = pc.Index(index_name)
index_names = [index['name'] for index in pc.list_indexes()]
index_names

['coffee-pages-index', 'rag-practice-easy-2', 'ragtest', 'rag-practice-easy']

# Upsert the data

In [70]:
print(data[0]['sparse_values'])
pc_index.upsert(data)

{'indices': [11, 202, 22, 139, 174, 115, 368],
 'values': [0.3203649566775846,
            0.27799258044110875,
            0.41149353572237,
            0.41149353572237,
            0.41149353572237,
            0.37786068859762806,
            0.41149353572237]}


UpsertResponse(upserted_count=104, _response_info={'raw_headers': {'date': 'Thu, 18 Dec 2025 16:35:32 GMT', 'content-type': 'application/json', 'content-length': '21', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '4', 'x-pinecone-request-logical-size': '169425', 'x-pinecone-request-latency-ms': '3799', 'x-pinecone-request-id': '4038503009702404605', 'x-envoy-upstream-service-time': '194', 'grpc-status': '0', 'server': 'envoy'}})

# Query the data

In [77]:
query = "How does fine‑tuning customize pretrained models?"
query_embedding = model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0].tolist()

In [78]:
result = pc_index.query(vector=query_embedding, top_k=3, include_metadata=True)
result.matches

[{'id': '63',
  'metadata': {'text': 'Fine-tuning customizes pretrained models'},
  'score': 0.907076836,
  'values': []},
 {'id': '21',
  'metadata': {'text': 'Hyperparameter tuning optimizes accuracy'},
  'score': 0.48522234,
  'values': []},
 {'id': '11',
  'metadata': {'text': 'Feature engineering improves model accuracy'},
  'score': 0.453323841,
  'values': []}]